In [1]:
import pandas as pd
import numpy as np

# 1. Define the Classic Golf Dataset (Categorical)
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 
                'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temp': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 
             'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 
                 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 
             'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'Play': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 
             'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
}
df = pd.DataFrame(data)

# 2. Core Math Functions
def calculate_entropy(target_col):
    """Calculates the entropy of a dataset column."""
    elements, counts = np.unique(target_col, return_counts=True)
    entropy = np.sum([(-counts[i]/np.sum(counts)) * np.log2(counts[i]/np.sum(counts)) for i in range(len(elements))])
    return entropy

def calculate_info_gain(data, split_feature, target_name="Play"):
    """Calculates Information Gain of splitting 'data' on 'split_feature'."""
    total_entropy = calculate_entropy(data[target_name])
    
    # Calculate weighted entropy of the split
    vals, counts= np.unique(data[split_feature], return_counts=True)
    weighted_entropy = np.sum([(counts[i]/np.sum(counts)) * calculate_entropy(data.where(data[split_feature]==vals[i]).dropna()[target_name]) 
                               for i in range(len(vals))])
    
    return total_entropy - weighted_entropy

# 3. The ID3 Recursive Algorithm
def id3(data, original_data, features, target_name="Play"):
    """Recursively builds the ID3 decision tree as a nested dictionary."""
    
    # Base Case 1: If all target values have the same value, return this pure node
    if len(np.unique(data[target_name])) <= 1:
        return np.unique(data[target_name])[0]
    
    # Base Case 2: If the dataset is empty, return the mode target feature value in the original dataset
    elif len(data) == 0:
        return np.unique(original_data[target_name])[np.argmax(np.unique(original_data[target_name], return_counts=True)[1])]
    
    # Base Case 3: If there are no more features to split on, return the mode target feature
    elif len(features) == 0:
        return np.unique(data[target_name])[np.argmax(np.unique(data[target_name], return_counts=True)[1])]
    
    # Recursive Branch: Grow the tree!
    else:
        # Find the feature with the highest Information Gain
        item_values = [calculate_info_gain(data, feature, target_name) for feature in features]
        best_feature_index = np.argmax(item_values)
        best_feature = features[best_feature_index]
        
        # Create the tree structure. The root gets the name of the best feature
        tree = {best_feature: {}}
        
        # Remove the feature with the best info gain from the feature list
        features = [i for i in features if i != best_feature]
        
        # Grow a branch under the root node for each possible value of the root node feature
        for value in np.unique(data[best_feature]):
            # Split the dataset
            sub_data = data.where(data[best_feature] == value).dropna()
            
            # Call the ID3 algorithm for each of those sub_datasets
            subtree = id3(sub_data, original_data, features, target_name)
            
            # Add the sub tree to the main tree
            tree[best_feature][value] = subtree
            
        return tree

# 4. Helper Function to Print the Nested Dictionary Beautifully
def print_tree(tree, indent=""):
    if not isinstance(tree, dict):
        print(f" -> Predict: [{tree}]")
        return

    for node_name, branches in tree.items():
        print()
        for branch_val, sub_tree in branches.items():
            print(f"{indent}Split on {node_name} == '{branch_val}'", end="")
            print_tree(sub_tree, indent + "    ")

# 5. Execute the Algorithm
features = df.columns[:-1].tolist()
custom_tree = id3(df, df, features)

print("--- Custom ID3 Multi-Way Decision Tree ---")
print_tree(custom_tree)
print("\n------------------------------------------")

--- Custom ID3 Multi-Way Decision Tree ---

Split on Outlook == 'Overcast' -> Predict: [Yes]
Split on Outlook == 'Rain'
    Split on Wind == 'Strong' -> Predict: [No]
    Split on Wind == 'Weak' -> Predict: [Yes]
Split on Outlook == 'Sunny'
    Split on Humidity == 'High' -> Predict: [No]
    Split on Humidity == 'Normal' -> Predict: [Yes]

------------------------------------------
